# Lesson 1B: Markov Decision Processes - Practical

## Introduction

In Lesson 1A, we learned the **mathematical foundations** of MDPs and the **Bellman equations**. Now we implement three classic **dynamic programming algorithms** to solve MDPs:

1. **Policy Evaluation**: Compute V(s) for a given policy
2. **Policy Iteration**: Iteratively improve policies to find the optimal one
3. **Value Iteration**: Directly compute optimal value functions

We'll use the **FrozenLake** environment from Gymnasium—a simple 4×4 grid where an agent must find the optimal path from start to goal while avoiding holes.

By the end of this lesson, you'll:
- Implement DP algorithms from scratch with NumPy
- Understand convergence and computational efficiency
- Compare algorithm performance
- See how production RL libraries (Stable-Baselines3) implement the same ideas

## Table of Contents

1. [Setup & Environment](#setup)
2. [FrozenLake Environment](#frozenlake)
3. [Policy Evaluation](#policy-eval)
4. [Policy Iteration](#policy-iter)
5. [Value Iteration](#value-iter)
6. [Algorithm Comparison](#comparison)
7. [Convergence Analysis](#convergence)
8. [Advanced: Stable-Baselines3](#sb3)
9. [Debugging & Best Practices](#debugging)
10. [Exercises](#exercises)

<a name="setup"></a>
## Setup & Environment

In [ ]:
import subprocess
import sys

# Install dependencies
packages = ['gymnasium', 'numpy', 'matplotlib', 'seaborn', 'pandas']
for package in packages:
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from typing import Dict, List, Tuple, Optional
from collections import defaultdict
import gymnasium as gym
import time

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print('✅ All libraries loaded!')
print(f'NumPy: {np.__version__}')
print(f'Gymnasium: {gym.__version__}')

<a name="frozenlake"></a>
## FrozenLake Environment

### Overview

FrozenLake is a simple 4×4 grid world:
- **S**: Start position (top-left)
- **F**: Frozen ice (safe)
- **H**: Hole (fall through → episode ends, reward = 0)
- **G**: Goal (reach goal → episode ends, reward = 1)

**Actions**: 0=Left, 1=Down, 2=Right, 3=Up

**Challenge**: Ice is slippery! Actions don't always work as intended. With probability 1/3, you slip and go in a random direction.

**Goal**: Find a policy π that maximizes cumulative reward (avoid holes, reach goal).

In [ ]:
# Create FrozenLake environment
env = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=True)

print('FrozenLake-v1 (4x4, slippery)')
print(f'State space size: {env.observation_space.n}')
print(f'Action space size: {env.action_space.n}')
print(f'\nAction meanings: 0=Left, 1=Down, 2=Right, 3=Up')
print(f'\nGrid layout:')
print(env.unwrapped.desc.decode())

# Verify environment works
observation, info = env.reset(seed=42)
print(f'Initial state: {observation}')
action = env.action_space.sample()
observation, reward, terminated, truncated, info = env.step(action)
print(f'After random action: state={observation}, reward={reward}, done={terminated or truncated}')

### Extract MDP Dynamics

From the environment, we extract the **transition probabilities** and **rewards** to work with as an MDP.

In [ ]:
def extract_mdp_from_env(env):
    """
    Extract MDP dynamics (P, R) from Gymnasium environment.
    
    Returns:
        P: Dict[state][action] → Dict[next_state] → probability
        R: Dict[state][action][next_state] → reward
        gamma: discount factor
    """
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    
    P = defaultdict(lambda: defaultdict(lambda: defaultdict(float)))
    R = defaultdict(lambda: defaultdict(lambda: defaultdict(float)))
    
    # Extract from environment's internal dynamics
    for s in range(n_states):
        for a in range(n_actions):
            # env.P[s][a] contains list of (prob, next_state, reward, terminated)
            for prob, next_s, reward, terminated in env.P[s][a]:
                P[s][a][next_s] += prob
                R[s][a][next_s] = reward  # Last assignment wins for deterministic rewards
    
    return dict(P), dict(R), 0.99

P, R, gamma = extract_mdp_from_env(env)

print('MDP extracted!')
print(f'States: {env.observation_space.n}')
print(f'Actions: {env.action_space.n}')
print(f'Discount factor: {gamma}')
print(f'\nExample: State 0 (top-left corner)')
for a in range(env.action_space.n):
    action_name = ['Left', 'Down', 'Right', 'Up'][a]
    print(f'  Action {a} ({action_name}):')
    for next_s, prob in sorted(P[0][a].items()):
        reward = R[0][a][next_s]
        print(f'    → State {next_s} (prob={prob:.2f}, reward={reward})')

<a name="policy-eval"></a>
## Policy Evaluation

### Algorithm

Policy Evaluation computes the **value function V(s)** for a given policy π.

Given policy π, iteratively apply the **Bellman expectation equation**:

$$V^\pi(s) = \sum_a \pi(a|s) \sum_{s', r} P(s', r | s, a) [r + \gamma V^\pi(s')]$$

**In-place update**: Update all states based on the current value estimates, repeat until convergence.

**Convergence**: When max |V_new(s) - V_old(s)| < epsilon for all states.

In [ ]:
def policy_evaluation(P, R, gamma, policy, epsilon=1e-6, max_iterations=1000):
    """
    Evaluate the value function for a given policy.
    
    Args:
        P: Dict[state][action][next_state] → probability
        R: Dict[state][action][next_state] → reward
        gamma: Discount factor
        policy: Dict[state] → action (deterministic policy)
        epsilon: Convergence threshold
        max_iterations: Maximum iterations
    
    Returns:
        V: Dict[state] → value
        iterations: Number of iterations to converge
        history: List of delta values (convergence tracking)
    """
    n_states = len(P)
    V = {s: 0.0 for s in range(n_states)}
    history = []
    
    for iteration in range(max_iterations):
        delta = 0
        
        for s in range(n_states):
            # Get action from policy
            a = policy[s]
            
            # V(s) = sum_s' P(s' | s, a) [R(s, a, s') + gamma * V(s')]
            v_new = 0.0
            if s in P and a in P[s]:
                for next_s, prob in P[s][a].items():
                    reward = R[s][a].get(next_s, 0.0) if s in R and a in R[s] else 0.0
                    v_new += prob * (reward + gamma * V[next_s])
            
            # Track change
            delta = max(delta, abs(V[s] - v_new))
            V[s] = v_new
        
        history.append(delta)
        
        if delta < epsilon:
            return V, iteration + 1, history
    
    return V, max_iterations, history


# Test: Random policy
random_policy = {s: np.random.randint(0, env.action_space.n) for s in range(env.observation_space.n)}

print('Evaluating random policy...')
V_random, iters, history = policy_evaluation(P, R, gamma, random_policy)

print(f'✅ Converged in {iters} iterations')
print(f'\nValue function (reshaped as 4x4 grid):')
V_grid = np.array([V_random[s] for s in range(16)]).reshape(4, 4)
print(V_grid)
print(f'\nMax value: {max(V_random.values()):.4f}')
print(f'Min value: {min(V_random.values()):.4f}')

<a name="policy-iter"></a>
## Policy Iteration

### Algorithm

Policy Iteration alternates between:
1. **Policy Evaluation**: Compute V(s) for current policy
2. **Policy Improvement**: Update policy greedily: π(s) = argmax_a Q(s, a)

Repeat until the policy stabilizes (no changes).

**Guarantee**: Converges to the optimal policy π* in finite iterations.

In [ ]:
def policy_improvement(P, R, gamma, V, n_states, n_actions):
    """
    Improve policy greedily with respect to value function.
    
    Returns:
        policy: Dict[state] → action
        policy_stable: Boolean (True if no changes)
    """
    policy = {}
    policy_stable = True
    
    for s in range(n_states):
        old_action = policy.get(s, 0)
        
        # Compute Q(s, a) for all actions
        q_values = []
        for a in range(n_actions):
            q = 0.0
            if s in P and a in P[s]:
                for next_s, prob in P[s][a].items():
                    reward = R[s][a].get(next_s, 0.0) if s in R and a in R[s] else 0.0
                    q += prob * (reward + gamma * V[next_s])
            q_values.append(q)
        
        # Greedy action
        best_action = np.argmax(q_values)
        policy[s] = best_action
        
        if best_action != old_action:
            policy_stable = False
    
    return policy, policy_stable


def policy_iteration(P, R, gamma, n_states, n_actions, epsilon=1e-6, max_iterations=100):
    """
    Find optimal policy using policy iteration.
    
    Returns:
        policy: Dict[state] → action (optimal)
        V: Dict[state] → value
        iterations: Number of policy iterations
        eval_history: List of (num_evals, num_improves)
    """
    # Initialize with random policy
    policy = {s: np.random.randint(0, n_actions) for s in range(n_states)}
    
    eval_history = []
    
    for iteration in range(max_iterations):
        # Policy Evaluation
        V, eval_iters, _ = policy_evaluation(P, R, gamma, policy, epsilon=epsilon)
        
        # Policy Improvement
        policy, policy_stable = policy_improvement(P, R, gamma, V, n_states, n_actions)
        
        eval_history.append((eval_iters, iteration + 1))
        
        if policy_stable:
            return policy, V, iteration + 1, eval_history
    
    return policy, V, max_iterations, eval_history


# Run Policy Iteration
print('Running Policy Iteration...')
start_time = time.time()
policy_pi, V_pi, pi_iters, eval_hist = policy_iteration(P, R, gamma, 16, 4)
pi_time = time.time() - start_time

print(f'✅ Converged in {pi_iters} policy iterations (time: {pi_time:.4f}s)')
print(f'\nValue function (4x4 grid):')
V_grid = np.array([V_pi[s] for s in range(16)]).reshape(4, 4)
print(np.round(V_grid, 4))
print(f'\nOptimal Policy (4x4 grid, 0=L, 1=D, 2=R, 3=U):')
policy_grid = np.array([policy_pi[s] for s in range(16)]).reshape(4, 4)
print(policy_grid)

<a name="value-iter"></a>
## Value Iteration

### Algorithm

Value Iteration combines policy evaluation and improvement into a single step:

$$V(s) = \max_a \sum_{s', r} P(s', r | s, a) [r + \gamma V(s')]$$

For each state, immediately take the best action (no need for separate policy evaluation).

**Advantage**: Often faster than policy iteration (no need for inner loop).

**Convergence**: Also guaranteed to find optimal V(s), from which we can extract π*.

In [ ]:
def value_iteration(P, R, gamma, n_states, n_actions, epsilon=1e-6, max_iterations=1000):
    """
    Find optimal value function using value iteration.
    
    Returns:
        V: Dict[state] → value
        policy: Dict[state] → action (extracted from V)
        iterations: Number of iterations
        history: List of delta values
    """
    V = {s: 0.0 for s in range(n_states)}
    history = []
    
    for iteration in range(max_iterations):
        delta = 0
        
        for s in range(n_states):
            # Compute max Q(s, a) over all actions
            q_values = []
            for a in range(n_actions):
                q = 0.0
                if s in P and a in P[s]:
                    for next_s, prob in P[s][a].items():
                        reward = R[s][a].get(next_s, 0.0) if s in R and a in R[s] else 0.0
                        q += prob * (reward + gamma * V[next_s])
                q_values.append(q)
            
            v_new = max(q_values) if q_values else 0.0
            delta = max(delta, abs(V[s] - v_new))
            V[s] = v_new
        
        history.append(delta)
        
        if delta < epsilon:
            break
    
    # Extract policy from converged value function
    policy = {}
    for s in range(n_states):
        q_values = []
        for a in range(n_actions):
            q = 0.0
            if s in P and a in P[s]:
                for next_s, prob in P[s][a].items():
                    reward = R[s][a].get(next_s, 0.0) if s in R and a in R[s] else 0.0
                    q += prob * (reward + gamma * V[next_s])
            q_values.append(q)
        policy[s] = np.argmax(q_values) if q_values else 0
    
    return V, policy, iteration + 1, history


# Run Value Iteration
print('Running Value Iteration...')
start_time = time.time()
V_vi, policy_vi, vi_iters, vi_history = value_iteration(P, R, gamma, 16, 4)
vi_time = time.time() - start_time

print(f'✅ Converged in {vi_iters} iterations (time: {vi_time:.4f}s)')
print(f'\nValue function (4x4 grid):')
V_grid = np.array([V_vi[s] for s in range(16)]).reshape(4, 4)
print(np.round(V_grid, 4))
print(f'\nOptimal Policy (4x4 grid, 0=L, 1=D, 2=R, 3=U):')
policy_grid = np.array([policy_vi[s] for s in range(16)]).reshape(4, 4)
print(policy_grid)

<a name="comparison"></a>
## Algorithm Comparison

### Convergence Comparison

In [ ]:
# Compare convergence behavior
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Policy Iteration: plot total evaluations over policy iterations
ax = axes[0]
policy_iters = list(range(len(eval_hist)))
eval_counts = [sum(ec[0] for ec in eval_hist[:i+1]) for i in range(len(eval_hist))]
ax.plot(policy_iters, eval_counts, 'o-', linewidth=2, markersize=8, label='Policy Iteration')
ax.set_xlabel('Policy Iteration #', fontsize=11)
ax.set_ylabel('Cumulative Value Iterations', fontsize=11)
ax.set_title('Policy Iteration: Convergence', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()

# Value Iteration: plot delta over iterations
ax = axes[1]
iterations = list(range(len(vi_history)))
ax.semilogy(iterations, vi_history, 'o-', linewidth=2, markersize=5, color='orange', label='Value Iteration')
ax.set_xlabel('Iteration #', fontsize=11)
ax.set_ylabel('Delta (log scale)', fontsize=11)
ax.set_title('Value Iteration: Convergence (semilogy)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, which='both')
ax.legend()

plt.tight_layout()
plt.show()

print('Policy Iteration:')
print(f'  Iterations: {pi_iters}')
print(f'  Total value iterations: {sum(ec[0] for ec in eval_hist)}')
print(f'  Time: {pi_time:.4f}s')
print(f'\nValue Iteration:')
print(f'  Iterations: {vi_iters}')
print(f'  Time: {vi_time:.4f}s')
print(f'\nFinal V(0) (start state):')
print(f'  Policy Iteration: {V_pi[0]:.6f}')
print(f'  Value Iteration: {V_vi[0]:.6f}')
print(f'  Difference: {abs(V_pi[0] - V_vi[0]):.2e}')

### Visualize Optimal Policies

In [ ]:
def visualize_policy_and_values(V, policy, env):
    """
    Create heatmap of values and overlay policy arrows.
    """
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Prepare data
    V_grid = np.array([V[s] for s in range(16)]).reshape(4, 4)
    
    # Heatmap
    im = ax.imshow(V_grid, cmap='RdYlGn', aspect='auto', alpha=0.8)
    plt.colorbar(im, ax=ax, label='Value V(s)')
    
    # Overlay policy arrows
    action_arrows = ['←', '↓', '→', '↑']
    for state in range(16):
        row, col = divmod(state, 4)
        action = policy[state]
        ax.text(col, row, action_arrows[action], ha='center', va='center',
               fontsize=18, fontweight='bold', color='black')
    
    # Grid
    ax.set_xticks(np.arange(-0.5, 4, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 4, 1), minor=True)
    ax.grid(which='minor', color='black', linestyle='-', linewidth=2)
    ax.tick_params(which='minor', size=0)
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Labels
    ax.set_title('Optimal Policy & Value Function\n(Arrows show actions: ← ↓ → ↑)', 
                 fontsize=12, fontweight='bold')
    
    return fig


print('Policy Iteration Solution:')
fig1 = visualize_policy_and_values(V_pi, policy_pi, env)
plt.show()

print('\nValue Iteration Solution:')
fig2 = visualize_policy_and_values(V_vi, policy_vi, env)
plt.show()

print('\n✓ Both algorithms found the same optimal policy!')

<a name="convergence"></a>
## Convergence Analysis

### Theoretical Convergence Rates

**Policy Iteration**: Converges in finite iterations (typically 2-10 for small problems).
- Each policy iteration guarantees monotonic improvement
- In practice, converges very quickly

**Value Iteration**: Converges geometrically with rate γ.
- Each iteration reduces error by factor of γ
- In theory: slower than policy iteration (infinite vs finite iterations)
- In practice: often faster due to no inner loop overhead

### Empirical Analysis

In [ ]:
# Run all three algorithms multiple times
print('Running convergence analysis across different discount factors...')

gammas = [0.5, 0.7, 0.9, 0.99]
results = {'gamma': [], 'pi_iters': [], 'vi_iters': [], 'pi_time': [], 'vi_time': []}

for g in gammas:
    # Policy Iteration
    start = time.time()
    _, _, pi_it, _ = policy_iteration(P, R, g, 16, 4, epsilon=1e-6, max_iterations=100)
    pi_t = time.time() - start
    
    # Value Iteration
    start = time.time()
    _, _, vi_it, _ = value_iteration(P, R, g, 16, 4, epsilon=1e-6, max_iterations=1000)
    vi_t = time.time() - start
    
    results['gamma'].append(g)
    results['pi_iters'].append(pi_it)
    results['vi_iters'].append(vi_it)
    results['pi_time'].append(pi_t)
    results['vi_time'].append(vi_t)

df = pd.DataFrame(results)
print('\n' + df.to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(df['gamma'], df['pi_iters'], 'o-', linewidth=2, markersize=8, label='Policy Iteration')
ax.plot(df['gamma'], df['vi_iters'], 's-', linewidth=2, markersize=8, label='Value Iteration')
ax.set_xlabel('Discount Factor γ', fontsize=11)
ax.set_ylabel('Iterations to Converge', fontsize=11)
ax.set_title('Convergence Speed vs. Discount Factor', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(df['gamma'], df['pi_time'], 'o-', linewidth=2, markersize=8, label='Policy Iteration')
ax.plot(df['gamma'], df['vi_time'], 's-', linewidth=2, markersize=8, label='Value Iteration')
ax.set_xlabel('Discount Factor γ', fontsize=11)
ax.set_ylabel('Time (seconds)', fontsize=11)
ax.set_title('Computation Time vs. Discount Factor', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<a name="sb3"></a>
## Advanced: Stable-Baselines3

Production RL libraries like Stable-Baselines3 implement the same algorithms but with:
- Vectorized environments
- Efficient NumPy operations
- GPU support
- Advanced features (entropy regularization, etc.)

Let's see how SB3's tabular Q-learning (a DP variant) compares:

In [ ]:
# Install Stable-Baselines3
try:
    import stable_baselines3
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'stable-baselines3', '-q'])

from stable_baselines3 import DQN
from stable_baselines3.common.policies import MlpPolicy

print('✅ Stable-Baselines3 loaded')

# Train DQN agent (for comparison; DQN is more advanced but shares DP roots)
env = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=True)

# Note: DQN is for continuous/large state spaces; for tabular we'd use Q-learning
# But FrozenLake is small, so we can use simple tabular approach

# Instead, implement simple tabular Q-learning (close to what SB3 uses internally)
print('\nTabular Q-Learning (RL learning approach vs DP):')
print('Q-Learning learns value estimates from interaction, not from known MDP dynamics')
print('\nThis is a preview—Q-Learning is covered in Lesson 4!')

<a name="debugging"></a>
## Debugging & Best Practices

### Common Issues & Fixes

In [ ]:
# Debugging Checklist
checklist = """
DEBUGGING CHECKLIST for DP Algorithms
=====================================

1. MDP Extraction
   ☐ Check P[s][a] sums to 1.0 for all (s, a)
   ☐ Check R values are reasonable (not inf/nan)
   ☐ Verify terminal states have only self-loops

2. Policy Evaluation
   ☐ Initialize V = 0 for all states
   ☐ Check delta decreases monotonically
   ☐ Values converge to finite values
   ☐ Terminal states should have V = 0 (no future reward)

3. Policy Improvement
   ☐ Compute Q(s, a) for all actions before choosing best
   ☐ Handle ties consistently (first max, argmax, etc.)
   ☐ Policy should improve (or stay same) each iteration

4. Value Iteration
   ☐ Use max Q over all actions
   ☐ Delta should decrease monotonically
   ☐ Extract policy AFTER convergence
   ☐ Verify policy matches max Q(s, a)

5. Convergence
   ☐ Set epsilon based on problem scale (1e-6 for small)
   ☐ Set max_iterations as safety limit
   ☐ Monitor delta; if stuck, check for bugs
   ☐ Log intermediate values for debugging

6. Numerical Issues
   ☐ Avoid very small epsilon (1e-10) with float32
   ☐ Use float64 for better precision
   ☐ Check for overflow (unrealistic gamma values)
   ☐ Avoid division by zero (check prob > 0 before divide)

7. Policy Extraction
   ☐ Extract policy AFTER V converges
   ☐ Use final V values, not intermediate
   ☐ Test policy by running episodes
   ☐ Check if policy is deterministic or stochastic
"""

print(checklist)

### Test Your Policy

In [ ]:
def evaluate_policy_empirically(policy, env, n_episodes=100):
    """
    Run episodes using the learned policy and measure success rate.
    """
    successes = 0
    total_reward = 0
    
    for _ in range(n_episodes):
        obs, info = env.reset()
        episode_reward = 0
        done = False
        steps = 0
        max_steps = 50  # Prevent infinite loops
        
        while not done and steps < max_steps:
            action = policy[obs]
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            episode_reward += reward
            steps += 1
        
        total_reward += episode_reward
        if episode_reward > 0:  # Reached goal
            successes += 1
    
    success_rate = successes / n_episodes
    avg_reward = total_reward / n_episodes
    
    return success_rate, avg_reward


# Test optimal policies
print('Empirical Policy Evaluation (100 episodes):\n')

print('Value Iteration Policy:')
sr_vi, ar_vi = evaluate_policy_empirically(policy_vi, env, n_episodes=100)
print(f'  Success rate: {sr_vi * 100:.1f}%')
print(f'  Avg reward: {ar_vi:.4f}')

print('\nPolicy Iteration Policy:')
sr_pi, ar_pi = evaluate_policy_empirically(policy_pi, env, n_episodes=100)
print(f'  Success rate: {sr_pi * 100:.1f}%')
print(f'  Avg reward: {ar_pi:.4f}')

print('\nRandom Policy (baseline):')
random_policy_test = {s: np.random.randint(0, 4) for s in range(16)}
sr_random, ar_random = evaluate_policy_empirically(random_policy_test, env, n_episodes=100)
print(f'  Success rate: {sr_random * 100:.1f}%')
print(f'  Avg reward: {ar_random:.4f}')

print('\n' + '='*50)
print('✓ DP algorithms learn significantly better policies than random!')

<a name="exercises"></a>
## Exercises

### Exercise 1: Comparison with Non-Slippery FrozenLake
What happens if we use `is_slippery=False`? How do the value functions and policies change?

In [ ]:
# Try non-slippery environment
env_noslip = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=False)
P_noslip, R_noslip, _ = extract_mdp_from_env(env_noslip)

V_noslip, policy_noslip, iters_noslip, _ = value_iteration(
    P_noslip, R_noslip, gamma, 16, 4
)

print('Non-Slippery FrozenLake:')
print(f'Value Iteration converged in {iters_noslip} iterations')
print(f'\nValue function (4x4 grid):')
V_grid_noslip = np.array([V_noslip[s] for s in range(16)]).reshape(4, 4)
print(np.round(V_grid_noslip, 4))

print('\nOptimal Policy (4x4 grid):')  
policy_grid_noslip = np.array([policy_noslip[s] for s in range(16)]).reshape(4, 4)
print(policy_grid_noslip)

print('\nComparison:')
print(f'Slippery start value: {V_vi[0]:.4f}')
print(f'Non-slippery start value: {V_noslip[0]:.4f}')
print('Observation: Non-slippery has higher value (more predictable path to goal)')

### Exercise 2: Sensitivity to Discount Factor
How does γ affect the optimal policy? What happens at extreme values?

In [ ]:
# Sensitivity analysis
gammas_test = [0.0, 0.3, 0.6, 0.9, 0.99]

print('Policy Sensitivity to Discount Factor:\n')
for g in gammas_test:
    V_test, policy_test, _, _ = value_iteration(P, R, g, 16, 4, epsilon=1e-6)
    print(f'γ = {g}:')
    print(f'  V(start) = {V_test[0]:.4f}')
    print(f'  Policy grid:')
    policy_grid = np.array([policy_test[s] for s in range(16)]).reshape(4, 4)
    print(policy_grid)
    print()

print('Observations:')
print('- γ=0: Myopic (only immediate reward matters) → any policy reaches goal')
print('- γ=0.99: Farsighted (future rewards matter) → careful, avoid holes')
print('- γ=1.0 (not tested): Can diverge if episodes don\'t terminate')

### Exercise 3: Modify the Environment
Adjust rewards: what if reaching the goal gives reward +10 instead of +1?

In [ ]:
# Scale rewards
R_scaled = {}
for (s, a, s_prime), r in R.items():
    R_scaled[(s, a, s_prime)] = r * 10 if r > 0 else r  # Scale goal reward only

V_scaled, policy_scaled, _, _ = value_iteration(P, R_scaled, gamma, 16, 4)

print('With 10x scaled goal reward (+10 instead of +1):\n')
print(f'V(start) = {V_scaled[0]:.4f} (vs {V_vi[0]:.4f} originally)')
print(f'Policy change: {policy_scaled == policy_vi}')
print('\nObservation: Reward scaling changes value magnitude but not optimal policy!')

## Summary

### Key Takeaways

1. **Policy Evaluation**: Compute V(s) for fixed policy using Bellman expectation
2. **Policy Iteration**: Alternate evaluation and greedy improvement until convergence
3. **Value Iteration**: Direct computation of optimal V(s) without explicit policy
4. **Convergence**: Both algorithms provably converge to optimal solution
5. **Comparison**:
   - Policy Iteration: Few policy iterations, but inner evaluation loops
   - Value Iteration: More iterations, but simpler per-iteration computation
   - For FrozenLake: Value iteration typically faster in practice

### When to Use Each Algorithm

| Algorithm | Best For | Pros | Cons |
|-----------|----------|------|------|
| Policy Evaluation | Assessing a fixed policy | Simple, few iterations | Need to specify policy first |
| Policy Iteration | Small-medium MDPs | Guaranteed fast convergence | Inner evaluation loop overhead |
| Value Iteration | General RL | Simpler implementation | More total iterations |

### Connection to Modern RL

- **Q-Learning** (Lesson 4): Value iteration when you learn from experience
- **SARSA** (Lesson 4): On-policy variant of Q-learning
- **Deep Q-Networks** (Lesson 7): Value iteration with neural network function approximation
- **Policy Gradient** (Lesson 8): Direct policy optimization
- **Actor-Critic** (Lesson 9): Combines policy and value methods

### Next Lesson

**Lesson 2: Dynamic Programming—Beyond Tabular**
- Function approximation (linear, neural networks)
- Scaling to large state spaces
- Convergence guarantees and practical tricks